# Ch5 GPT 组装 教案

**课程名称：** GPT 组装：从概率预测到文本生成

**预计总时长：** 90-100 分钟

**源文件：** `Ch5_GPT_Assembly/Ch5_GPT_Assembly.ipynb`（共 24 个 Cell，Cell 0-23）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 00:00-08:00 | 开场、前置回顾与环境准备 | Cell 0-5 | 8 分钟 |
| 08:00-25:00 | 正弦位置编码（Sinusoidal PE） | Cell 6-7 | 17 分钟 |
| 25:00-45:00 | RoPE 旋转位置编码 | Cell 8-11 | 20 分钟 |
| 45:00-50:00 | 休息 + 回顾 | -- | 5 分钟 |
| 50:00-65:00 | 完整 GPT 模型组装 | Cell 12-14 | 15 分钟 |
| 65:00-82:00 | 文本生成：采样策略 | Cell 15-19 | 17 分钟 |
| 82:00-87:00 | 休息 + 回顾 | -- | 5 分钟 |
| 87:00-95:00 | 练习：实现 Temperature + Top-k 采样 | Cell 23 | 8 分钟 |
| 95:00-100:00 | 总结与下一章预告 | Cell 20-22 | 5 分钟 |

---

## 课前准备

- [ ] 确认 PyTorch 已安装（源 notebook 使用 PyTorch 2.10.0+cu128）
- [ ] 确认 matplotlib、numpy 可用
- [ ] 确认中文字体设置正确（Microsoft YaHei / SimHei），以及 `../assets/fonts/NotoSansCJKsc-Regular.otf` 字体文件存在
- [ ] 提前运行一遍全部 Cell，确认无报错（特别是 Cell 5 的架构全景图和 Cell 10 的 RoPE 可视化）
- [ ] 准备白板/画板用于画 GPT 架构分层图
- [ ] 回顾 Ch4 Transformer Block 的内容，确保能自如衔接

---

## 第一段：开场、前置回顾与环境准备（Cell 0-5）

📍 运行 Cell 4（环境导入）、Cell 5（GPT 架构全景图）

⏱ 时间分配：8 分钟（开场引导 2 分钟 + 前置回顾 2 分钟 + Cell 4 运行 1 分钟 + Cell 5 架构图讲解 3 分钟）

🎯 本段目标
- 回顾前四章的「积木」，建立组装完整模型的学习动机
- 确认环境就绪
- 通过架构全景图让学生对 GPT 结构有全局认知

🗣 讲课话术

> 大家好，前面四章我们一直在造「积木」——Ch1 的 Autograd、Ch2 的 Embedding、Ch3 的 Self-Attention、Ch4 的 Transformer Block。今天我们终于要把这些积木拼成一个完整的建筑了——GPT。
>
> GPT 的全称是 Generative Pre-trained Transformer。名字拆开看：Generative 是说它能生成文本，Pre-trained 是说它在大量文本上预训练，Transformer 就是我们前几章学的核心架构。
>
> 但在组装之前，我们有一个关键问题要解决。大家还记得 Ch3 讲 Self-Attention 的时候提过一个问题吗？——Attention 是「置换不变的」。也就是说，「狗咬人」和「人咬狗」对 Attention 来说是一样的！这在语言里显然不行，词序至关重要。
>
> 所以今天第一个话题就是「位置编码」——怎么告诉模型每个词排在第几位。
>
> 先跑一下 Cell 4 确认环境。然后看 Cell 5 这张 GPT 架构全景图。
>
> 大家看这张图，从下往上：输入 Token 进来，先过 Token Embedding 加上 Position Embedding，然后经过 N 个 Transformer Block（这就是 Ch4 的积木重复堆叠），最后 Final LayerNorm 再接一个 LM Head 输出 Logits。Logits 就是模型对词表里每个词的「打分」，再经过采样策略就能生成下一个词。
>
> 今天我们就沿着这条线，从位置编码到 GPT 组装，再到文本生成的采样策略，一路走完。

👀 输出要点
- Cell 4 应输出：`PyTorch version: 2.10.0+cu128`
- Cell 5 输出一张 GPT 架构全景图（16x12），展示从 Input Tokens 到 Output 的完整数据流，底部打印：
  - `GPT 关键组件：`
  - `1. Token + Position Embedding -> 注入位置信息`
  - `2. N x Transformer Block -> 提取语义特征`
  - `3. 每个Block: LayerNorm -> Attention -> 残差 -> LayerNorm -> FFN -> 残差`
  - `4. LM Head -> 预测词表中每个词的概率`

❓ 预判问题

Q: GPT 和 BERT 有什么区别？
A: 核心区别在于 Attention 的方向——GPT 用因果掩码（Causal Mask），只能看过去的 token，适合生成任务；BERT 用双向 Attention，能看所有 token，适合理解任务。我们 Ch3 讲过这个区别。

Q: LM Head 是什么？
A: 就是一个线性层，把 Transformer 输出的隐藏状态（维度 d_model）映射到词表大小（vocab_size），输出每个词作为「下一个词」的分数。后面 Cell 14 会看到具体实现。

➡️ 转场

> 好，全景图看完了。现在我们深入第一个关键组件——位置编码。先从经典的正弦位置编码开始。

---

## 第二段：正弦位置编码 Sinusoidal PE（Cell 6-7）

📍 运行 Cell 7（SinusoidalPositionalEncoding 类 + 可视化热力图）

⏱ 时间分配：17 分钟（理论 8 分钟 + 代码讲解与 TODO 补全 5 分钟 + 可视化分析 4 分钟）

🎯 本段目标
- 理解为什么 Attention 需要位置编码（置换不变性问题）
- 掌握正弦位置编码的公式及其设计动机
- 理解「不同频率 = 不同时钟」的直觉
- 完成 TODO：填写余弦维度的编码

🗣 讲课话术

> Cell 6 的 Markdown 解释了正弦位置编码的公式。我先帮大家翻译成直觉。
>
> 想象你有一排钟表，每个钟表的指针转速不同。最左边的钟表像秒针，转得很快——每个位置变化都很大；最右边的像时针，转得很慢——隔好多个位置才能看出变化。
>
> 公式里的 `10000^(2i/d_model)` 就是控制转速的。当 i=0 时（低维度），除数是 1，频率最高；当 i=d_model/2 时（高维度），除数是 10000，频率极低。
>
> sin 和 cos 成对出现不是巧合——它们组合起来可以表达任意相位的正弦波。这意味着什么呢？意味着 PE(pos+k) 可以表示为 PE(pos) 的线性变换，变换矩阵只依赖偏移量 k，不依赖绝对位置 pos。这给了模型表达「相对位置」的能力。
>
> 好，来看代码。Cell 7 的 `SinusoidalPositionalEncoding` 类里有一个 TODO——奇数维度用 cos。大家看偶数维度已经写好了：`pe[:, 0::2] = torch.sin(position * div_term)`。奇数维度怎么写？
>
> 没错，就是 `pe[:, 1::2] = torch.cos(position * div_term)`。输入参数完全一样，只是 sin 换成 cos。
>
> 注意 `self.register_buffer('pe', pe)` 这行——buffer 意味着这个张量会随模型保存和加载，但不会被优化器更新。因为位置编码是固定的，不需要学习。
>
> 运行 Cell 7 看可视化。横轴是位置 0-99，纵轴是维度 0-63。颜色是编码值。
>
> 大家注意看：底部（低维度）条纹很密，像秒针转得快；顶部（高维度）条纹很宽，像时针转得慢。每个位置的纵向切面就是一个独特的「指纹」——不同频率的 sin/cos 值的组合。
>
> 底部打印的那句话说得很好：「每个位置有唯一的编码模式！」

👀 输出要点
- Cell 7 图像：100x64 的热力图，使用 RdBu_r 色图，横轴「位置 (Position)」、纵轴「维度 (Dimension)」
- 低维度（底部）条纹密集变化快，高维度（顶部）条纹稀疏变化慢
- 底部文字：`每个位置有唯一的编码模式！`

❓ 预判问题

Q: 为什么底数是 10000 而不是其他数？
A: Cell 6 有解释——底数决定了最长波长。最低频率维度的波长为 2pi * 10000，足以覆盖绝大多数序列长度。底数越大，频率分布越平缓，长序列上的区分能力越好。10000 是原始论文的经验选择。

Q: `register_buffer` 和 `nn.Parameter` 有什么区别？
A: Parameter 会被优化器更新（可学习），buffer 不会。位置编码是固定的公式计算结果，不需要学习，所以用 buffer。

Q: 这个位置编码是加到 Embedding 上的？
A: 对，`forward` 里是 `return x + self.pe[:, :x.size(1), :]`——直接逐元素相加。所以位置编码和词嵌入的维度必须一样。

➡️ 转场

> 正弦位置编码是 2017 年原始 Transformer 论文提出的。但现在的大模型——LLaMA、Mistral、Qwen——全部换成了 RoPE。为什么？我们接下来就看。

---

## 第三段：RoPE 旋转位置编码（Cell 8-11）

📍 运行 Cell 9（precompute_freqs_cis + apply_rotary_emb 函数定义与测试）、Cell 10（RoPE 可视化）；浏览 Cell 11（Sin/Cos 加法式 vs RoPE 的数学对比 Markdown）

⏱ 时间分配：20 分钟（理论与动机 7 分钟 + 代码讲解与 TODO 5 分钟 + 可视化 3 分钟 + Sin/Cos vs RoPE 对比 5 分钟）

🎯 本段目标
- 理解 RoPE 的核心思想：不是「加」位置，而是「旋转」Q/K
- 掌握旋转使点积只依赖相对位置 m-n 的数学证明
- 完成两个 TODO：xk_ 的复数转换和 xk_out 的旋转计算
- 理解 RoPE 优于正弦编码的四个原因

🗣 讲课话术

> RoPE 是 Su et al. 2021 年提出的，全称 Rotary Position Embedding——旋转位置编码。这个名字很形象。
>
> 正弦位置编码是「加法」：词向量 + 位置向量。RoPE 是「乘法」：在算 Attention 的时候，把 Q 和 K 旋转一个角度。
>
> 为什么要旋转？核心洞察是：如果 Q 在位置 m 旋转了 m*theta 度，K 在位置 n 旋转了 n*theta 度，那它们的点积——也就是 Attention 分数——只依赖于 (m-n)*theta。绝对位置 m 和 n 消失了，剩下的只有相对距离 m-n！
>
> 我用一个钟表的比喻来理解。想象每个 token 手上拿着一根指针。位置 0 的指针指向 12 点方向，位置 1 旋转到 1 点方向，位置 2 旋转到 2 点方向......两个 token 之间的「夹角」就是它们的相对距离。不管绝对位置是多少，只要相对距离一样，夹角就一样。
>
> Cell 8 的 Markdown 里有一个二维的数学证明。关键的一步是利用旋转矩阵的正交性：R(alpha)^T = R(-alpha)。所以 (R(m)q)^T (R(n)k) = q^T R(-m) R(n) k = q^T R(n-m) k。结果只依赖 n-m。
>
> 代码上，RoPE 有一个优雅的复数实现。把每对维度 (q_{2i}, q_{2i+1}) 看成一个复数 q_{2i} + j*q_{2i+1}，旋转就是乘以 e^{j*m*theta}。这就是 Cell 9 里 `torch.polar` 和 `torch.view_as_complex` 的含义。
>
> Cell 9 有两个 TODO。第一个是 `xk_` 的计算，和 `xq_` 完全一样——把 xk 转成复数形式。第二个是 `xk_out`，和 `xq_out` 一样——把复数乘以频率后转回实数。
>
> 运行看输出：`频率形状: torch.Size([100, 32])`。100 是最大序列长度，32 是 dim/2=64/2——因为每对维度共享一个频率。
>
> 现在看 Cell 10 的可视化。左图是各位置的旋转相位——用 twilight 色图，你可以看到低维度（底部）从紫到黄变化很快，高维度（顶部）几乎不变。右图是相邻位置的相位差——对于每个维度，相邻位置的相位差是常数。这就是「匀速旋转」的体现。
>
> 最后看 Cell 11，这是这一段的精华——为什么加法式的 Sin/Cos 不如 RoPE？
>
> 加法式把位置向量加到输入上，进入 Q/K 计算后展开得到四项：内容-内容、内容-位置、位置-内容、位置-位置。第一项是我们想要的，后面三项是「噪声」。
>
> RoPE 是乘法作用在 Q/K 上，点积展开后只有一项：q^T R(n-m) k——干干净净，只有内容信息和相对位置信息。
>
> Cell 11 还提到了一个钟表的比喻：不同维度像不同转速的指针，秒针转得快捕捉短程关系，时针转得慢捕捉长程关系。这跟正弦编码的多频率思想是一脉相承的。

👀 输出要点
- Cell 9 输出：`频率形状: torch.Size([100, 32])` 和 `RoPE 本质是在复数空间中旋转向量！`
- Cell 10 输出：并排两张图——左图「RoPE 各位置的旋转相位」(twilight 色图)，右图「相邻位置的相位差」(RdBu_r 色图)
- 左图中低维度变化快（彩色条纹密集），高维度变化慢（颜色接近均匀）
- 右图中每个维度的相位差接近常数（颜色接近均匀的水平带）

❓ 预判问题

Q: `torch.polar` 和 `torch.view_as_complex` 分别在做什么？
A: `torch.polar(abs, angle)` 生成复数 abs * e^{j*angle}，用于预计算旋转因子。`torch.view_as_complex` 把实数张量的最后一维（必须是 2）解读为复数的实部和虚部，不复制数据。

Q: RoPE 有额外参数需要学习吗？
A: 没有！所有旋转角度由公式直接计算，theta_i = 1/10000^{2i/d}。这是 RoPE 的优点之一——零额外参数。

Q: RoPE 能处理超出训练长度的序列吗？
A: 原始 RoPE 在超出训练长度后效果会退化，但配合 NTK-aware 缩放、YaRN 等技术可以外推到更长长度。这比正弦编码的外推能力要好得多。Cell 8 的 Markdown 提到了这一点。

Q: 为什么 RoPE 只作用在 Q 和 K 上，不作用在 V 上？
A: 因为位置信息是通过 Q 和 K 的点积来影响注意力分数的。V 是实际被加权求和的「内容」，不需要位置旋转。

➡️ 转场

> 位置编码搞定了。现在我们手里有了所有积木：Token Embedding、位置编码、Transformer Block。是时候组装完整的 GPT 了！先休息 5 分钟。

---

## 休息 + 回顾（第 45-50 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. Self-Attention 是置换不变的，需要位置编码来告诉模型词的顺序。正弦位置编码用不同频率的 sin/cos 给每个位置一个独特「指纹」，加到词向量上。
2. RoPE 不是加位置信息，而是在计算 Attention 时旋转 Q 和 K 向量，使点积天然只依赖相对位置 m-n——没有内容-位置交叉项的干扰。
3. 现代大模型（LLaMA、Mistral、Qwen）全部使用 RoPE，因为它无额外参数、纯粹编码相对位置、具有距离衰减的归纳偏置、长度外推能力更好。

**下一段预告：**

> 接下来我们把 Token Embedding、Position Embedding、Transformer Block 和 LM Head 全部拼起来，组装一个完整的 GPT 模型。参数量只有 937,728——一个「迷你 GPT」。

---

## 第四段：完整 GPT 模型组装（Cell 12-14）

📍 运行 Cell 13（CausalSelfAttention + MLP + Block 类定义）、Cell 14（GPT 类定义 + 实例化）

⏱ 时间分配：15 分钟（Cell 12 Markdown 浏览 1 分钟 + Cell 13 代码讲解 5 分钟 + Cell 14 GPT 类讲解 7 分钟 + 配置分析 2 分钟）

🎯 本段目标
- 理解 GPT 的完整数据流：idx -> Embedding -> N x Block -> LayerNorm -> LM Head -> logits
- 理解权重共享（Weight Tying）的原理和好处
- 理解参数初始化的策略
- 验证模型参数量 937,728

🗣 讲课话术

> Cell 13 先定义了三个组件类。这些我们在 Ch3 和 Ch4 都见过，这里只是复习和整合。
>
> `CausalSelfAttention`：带因果掩码的自注意力。注意 `self.register_buffer("mask", torch.tril(...))`——用 `torch.tril` 生成下三角矩阵做掩码，和 Ch3 讲的一模一样。`forward` 里 Q/K/V 是通过一个大 Linear 层 `c_attn` 一次算出来的（3 * n_embd），然后 split。这比三个独立的 Linear 更高效。
>
> `MLP`：两层全连接，中间是 GELU 激活。注意中间维度是 4 * n_embd——这是 Transformer 的标准设计，FFN 的隐藏层是嵌入维度的 4 倍。
>
> `Block`：Pre-Norm 结构——先 LayerNorm 再 Attention/MLP，加残差连接。和 Ch4 完全一致。
>
> 现在看重头戏——Cell 14 的 `GPT` 类。`__init__` 里用 `nn.ModuleDict` 组织了所有组件：
> - `wte`：Token Embedding，形状 [vocab_size, n_embd] = [1000, 128]
> - `wpe`：Position Embedding，形状 [block_size, n_embd] = [128, 128]
> - `h`：4 个 Transformer Block 组成的 ModuleList
> - `ln_f`：最终的 LayerNorm
>
> 然后 `lm_head`：Linear(128, 1000)，把隐藏状态映射到词表大小。
>
> 这里有个非常重要的设计：`self.transformer.wte.weight = self.lm_head.weight`——权重共享！Token Embedding 矩阵是 [1000, 128]，LM Head 矩阵也是 [1000, 128]（因为 Linear 的 weight 形状是 [out, in]），两者直接共享同一个张量。
>
> 为什么能共享？因为 Embedding 是把 token ID 映射到向量空间，LM Head 是把向量空间映射回 token 概率。它们是「互逆」的操作，用同一套「词典」是合理的。而且省了 128,000 个参数！
>
> `forward` 的流程很清晰：
> 1. Token Embedding + Position Embedding（注意这里用的是可学习的位置嵌入，不是 Sinusoidal 也不是 RoPE）
> 2. Dropout
> 3. 依次过 4 个 Transformer Block
> 4. Final LayerNorm
> 5. LM Head 输出 logits [B, T, vocab_size]
> 6. 如果有 targets，计算 cross_entropy loss
>
> 看输出：`GPT 参数量: 937,728`。不到 100 万参数，真正的迷你 GPT。GPT-2 Small 是 1.17 亿，GPT-3 是 1750 亿。我们这个大概是 GPT-2 的 1/125。
>
> 初始化策略：所有 Linear 和 Embedding 用均值 0、标准差 0.02 的正态分布。0.02 这个数字来自 GPT-2 的论文，是经验值。

👀 输出要点
- Cell 13 无输出（纯类定义）
- Cell 14 输出：`GPT 参数量: 937,728`
- 配置参数：vocab_size=1000, block_size=128, n_layer=4, n_head=4, n_embd=128, dropout=0.1

❓ 预判问题

Q: 权重共享为什么能减少参数？不是同一个矩阵吗？
A: 如果不共享，Embedding 矩阵 [1000, 128] 有 128,000 个参数，LM Head [128, 1000] 又有 128,000 个——总共 256,000。共享后只算一次 128,000，省了一半。

Q: 这里用了可学习的位置嵌入 `nn.Embedding(block_size, n_embd)`，跟前面讲的 Sinusoidal 和 RoPE 有什么关系？
A: 这里为了代码简洁用了最简单的可学习位置嵌入。实际生产模型（如 LLaMA）会用 RoPE。前面讲 Sinusoidal 和 RoPE 是让大家理解原理，这里是展示完整 GPT 架构。

Q: `nn.ModuleDict` 和普通的 dict 有什么区别？
A: ModuleDict 会注册子模块，让 PyTorch 能追踪参数、移动设备（.to(device)）、保存/加载模型。普通 dict 里的模块是「不可见」的。

Q: cross_entropy loss 为什么要 view(-1)？
A: `F.cross_entropy` 要求输入形状是 [N, C]（N 个样本，C 个类别）。logits 原来是 [B, T, vocab_size]，view(-1, vocab_size) 把 batch 和序列长度展平成一维。targets 也同理展平。

➡️ 转场

> 模型组装完毕！但一个有趣的问题是：模型输出了 logits（对词表每个词的打分），怎么从这些分数里选出下一个词？直接选最大的？随机选？这就是采样策略的学问。

---

## 第五段：文本生成——采样策略（Cell 15-19）

📍 运行 Cell 16（generate 函数定义）、Cell 17（Temperature 可视化）、Cell 18（Top-k / Top-p 可视化）、Cell 19（生成测试）

⏱ 时间分配：17 分钟（Cell 15 理论 5 分钟 + Cell 16 generate 函数 3 分钟 + Cell 17 Temperature 可视化 3 分钟 + Cell 18 Top-k/Top-p 可视化 3 分钟 + Cell 19 生成演示 3 分钟）

🎯 本段目标
- 理解 Temperature 对概率分布的影响（控制熵/不确定性）
- 理解 Top-k 的局限和 Top-p 的自适应优势
- 通过可视化直观看到各策略的差异
- 理解自回归生成的完整流程

🗣 讲课话术

> Cell 15 的 Markdown 讲了三种采样策略，我用一个比喻串起来。
>
> 假设你在餐厅点菜，菜单上有 1000 道菜（词表大小 1000），模型给每道菜打了分。
>
> **Temperature** 就像你的冒险精神。Temperature 趋近 0 时，你永远点评分最高的那道菜——安全但无聊。Temperature 很高时，你什么菜都可能点——冒险但可能踩雷。Temperature=1 是模型的「原始判断」。
>
> 数学上，Temperature 就是在 softmax 之前除以 T：`p_i = exp(z_i/T) / sum(exp(z_j/T))`。T 小，logits 被放大，分布更「尖」；T 大，logits 被压缩，分布更「平」。
>
> 运行 Cell 17 看效果。四张柱状图，从左到右 Temperature=0.5/1.0/1.5/2.0。T=0.5 时 Token 0 的概率是 0.47，远超其他；T=2.0 时最高也才 0.20，几乎平均分配。
>
> **Top-k** 是说「我只从评分前 k 名的菜里选」。问题是 k 固定不灵活——有时候前 3 名就占了 90% 的概率，有时候前 50 名才凑到 90%。
>
> **Top-p（核采样）** 更聪明：「我从概率累加到 p 的那些菜里选」。概率集中时可能只有 2-3 个候选，概率分散时可能有几百个。它自动适应分布的「宽窄」。
>
> Cell 15 的 Markdown 里有个具体数值例子。logits=[2.0, 1.5, 1.0, 0.5, 0.1, -1.0]，对应 6 个 token。不同 Temperature 的概率表格很直观——T=0.5 时 Token A 概率 0.506，T=2.0 时只有 0.210。Top-k=3 保留 A/B/C，Top-p=0.9 需要累加到第 5 个 token 才够 0.9。
>
> 运行 Cell 18 看 Top-k 和 Top-p 的可视化对比。绿色是保留的 token，灰色是被过滤掉的。Top-k=3 固定保留 3 个，Top-p=0.8 保留的数量取决于分布集中度。
>
> Cell 16 是 generate 函数。核心是一个循环：每轮取最后一个位置的 logits，做 Temperature 缩放、Top-k/Top-p 过滤、softmax、multinomial 采样，然后把新 token 拼到序列末尾。这就是「自回归」——用已生成的 token 预测下一个。
>
> Cell 19 用我们未训练的模型做了一次生成演示。初始序列 [607, 484, 901, 954, 177]。看 Greedy（temp=0.1）的结果，很多重复——678, 200, 588...；Random（temp=1.5）则完全随机。这些都是无意义的数字，因为模型没训练过！但生成流程本身是完整的。
>
> 训练后，这些数字就会变成有意义的 token ID，映射回文本就是通顺的句子了。

👀 输出要点
- Cell 16 输出：`生成函数定义完成！`
- Cell 17 图像：4 张并排柱状图，Temperature=0.5/1.0/1.5/2.0，概率分布从尖锐到平坦。底部输出：
  - `Temperature:`
  - `  低 (0.5): 更确定性，选择高概率 token`
  - `  高 (2.0): 更随机，概率分布更平坦`
- Cell 18 图像：3 张并排柱状图（原始分布 / Top-k=3 / Top-p=0.8），绿色=保留，灰色=过滤。底部输出：
  - `Top-k: 只保留概率最高的 k 个（绿色）`
  - `Top-p: 保留累积概率达到 p 的 token（绿色）`
- Cell 19 输出：初始序列 [607, 484, 901, 954, 177]，四种策略生成的 15 个 token 序列
  - Greedy (temp=0.1): `[607, 484, 901, 954, 177, 678, 200, 588, 701, 820, 476, 514, 102, 721, 637]`
  - Random (temp=1.5): `[607, 484, 901, 954, 177, 315, 821, 824, 52, 901, 467, 135, 356, 19, 622]`
  - Top-k (k=5): 注意可能有重复 token（如 701, 701 和 496, 496, 496）
  - Top-p (p=0.9): 多样性介于 Greedy 和 Random 之间
  - 底部提醒：`注意：模型未训练，生成的是随机数字`

❓ 预判问题

Q: ChatGPT 的 Temperature 参数和这里是同一个东西吗？
A: 原理完全一样！ChatGPT API 的 temperature 参数就是控制采样时 softmax 的「锐度」。设成 0 就是 greedy（确定性），设成 1.5-2.0 就更有创造力但也更容易「胡说八道」。

Q: Top-k 和 Top-p 可以同时用吗？
A: 可以！Cell 16 的 generate 函数就支持同时设置。实践中很多模型会同时用 Top-p=0.9 和 Top-k=50，取两者的交集。

Q: 为什么 Top-p 过滤时 `sorted_indices_to_remove[..., 0] = 0`？
A: 确保至少保留概率最高的那个 token。否则如果 Top-p 设得很小（比如 0.01），可能所有 token 都被过滤掉，导致没有候选。

Q: 为什么要截断到 block_size？
A: 因为模型的位置嵌入最多支持 block_size=128 个位置。序列更长就超出了位置嵌入的范围。真实模型会用滑动窗口或 RoPE 来处理更长的上下文。

➡️ 转场

> 采样策略讲完了。先休息 5 分钟，然后我们做一个动手练习——自己实现 Temperature + Top-k 采样函数。

---

## 休息 + 回顾（第 82-87 分钟）

⏱ 时间分配：5 分钟

**三句话回顾：**

1. GPT 架构的数据流：Token ID → Token Embedding + Position Embedding → N 个 Transformer Block → Final LayerNorm → LM Head → Logits → 采样。参数量 937,728，其中 Token Embedding 和 LM Head 共享权重。
2. Temperature 控制概率分布的「尖锐/平坦」程度——低温更确定（像 greedy），高温更随机（像均匀分布）。
3. Top-k 固定候选数量，Top-p 固定累积概率阈值——Top-p 能自动适应不同上下文的分布宽窄，更灵活。

**下一段预告：**

> 接下来大家动手实现一个 `sample_next_token` 函数，把 Temperature 和 Top-k 的逻辑亲手写一遍。

---

## 第六段：练习——实现 Temperature + Top-k 采样（Cell 23）

📍 运行 Cell 23（sample_next_token 函数 + 测试）

⏱ 时间分配：8 分钟

🎯 本段目标
- 学生独立实现 4 步采样流程：Temperature 缩放 → Top-k 过滤 → Softmax → Multinomial 采样
- 通过三种不同设置验证采样行为的差异

🗣 讲课话术

> Cell 23 有一个 `sample_next_token` 函数，4 个 TODO 要补全。logits 是 7 个 token 的打分 [2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0]，对应词表 ["the", "cat", "sat", "on", "a", "big", "mat"]。
>
> 大家先自己试，2 分钟后我给第一个提示。

### Hint 节奏

**0-2 分钟：** 自己尝试，不给提示。

**2 分钟第一个提示：**
> Step 1 最简单：`logits = logits / temperature`。一行代码。Step 3 也是一行：`probs = F.softmax(logits, dim=-1)`。

**4 分钟关键代码：**
> Step 2 Top-k 过滤的思路：用 `torch.topk` 找到第 k 大的值作为阈值，然后把小于阈值的 logits 设为负无穷。
> ```python
> top_k_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
> threshold = top_k_values[-1]
> logits[logits < threshold] = float('-inf')
> ```
> Step 4 采样：`next_token = torch.multinomial(probs, num_samples=1)`

### 常见错误

1. **Temperature 除反了**：写成 `logits = logits * temperature` 而不是除以 temperature。记住：temperature 越高越随机，所以是除法让 logits 变小。
2. **Top-k 用 sort 而不是 topk**：`torch.topk` 比 sort 更高效，只找前 k 个不需要完整排序。
3. **忘记 min(top_k, logits.size(-1))**：如果 k 比词表还大，`topk` 会报错。
4. **multinomial 后忘记 .item()**：返回的是 tensor，要用 `.item()` 转成 Python int。

### 验证标准

运行后应看到三组输出：
- **Temperature=0.3 (确定性高)**: 几乎全是 "the"（概率最高的 token），偶尔出现 "cat"
- **Temperature=2.0 (随机性高)**: 各种词都有可能出现，分布较均匀
- **Top-k=3**: 只出现 "the"、"cat"、"sat" 三个词

源 notebook 的输出示例：
```
Temperature=0.3: → the → the → the → the → cat
Temperature=2.0: → a → the → on → a → sat
Top-k=3:         → the → sat → sat → the → cat
```

👀 输出要点
- Temperature=0.3 几乎只选 "the"（验证了低温度 = 高确定性）
- Temperature=2.0 出现了 "a"、"on"、"sat" 等低概率词（验证了高温度 = 高随机性）
- Top-k=3 严格只在 "the"、"cat"、"sat" 中选择（验证了 Top-k 过滤生效）

❓ 预判问题

Q: 为什么 Temperature=0.3 不是 100% 选 "the"？
A: Temperature=0.3 时 "the" 的概率大约 77%，"cat" 约 19%。只有 Temperature 趋近 0 时才完全退化为 argmax。0.3 已经很低但还没到极限。

Q: `torch.multinomial` 是什么？
A: 按给定的概率分布随机采样。`probs = [0.5, 0.3, 0.2]`，multinomial 有 50% 概率返回 0，30% 返回 1，20% 返回 2。

➡️ 转场

> 练习做完了，最后我们花几分钟总结全章。

---

## 第七段：总结与下一章预告（Cell 20-22）

📍 浏览 Cell 20（总结 Markdown）、Cell 21（Extra 思考题）、Cell 22（下一步链接）

⏱ 时间分配：5 分钟

🎯 本段目标
- 总结全章核心概念
- 提示面试高频题
- 引出 Ch6 Tokenizer

🗣 讲课话术

> Cell 20 有一张完整的 ASCII 架构图和公式速查表，大家课后对照着复习。
>
> 三道面试高频题，我快速过一遍：
>
> **Q1：权重共享是什么？** Token Embedding [V, d] 和 LM Head [d, V] 互为转置，共享权重减少参数，同时让输入和输出端的词表示保持一致。
>
> **Q2：RoPE 比 Sin/Cos 好在哪？** RoPE 是乘法作用于 Q/K，点积天然只依赖相对位置 m-n，没有交叉噪声项。正弦编码是加法作用于输入，展开后有内容-位置的混合干扰项。
>
> **Q3：Top-p 比 Top-k 好在哪？** Top-k 固定 k 个候选，Top-p 根据累积概率阈值动态调整候选数量，能自适应不同上下文的分布宽窄。
>
> Cell 21 有三道 Extra 思考题，大家课后挑战。尤其第二道——Temperature=0.5 相当于把 logits 放大 2 倍再 softmax，为什么分布更尖锐？大家可以从指数函数的性质来推导。
>
> 下一章 Ch6 Tokenizer——模型只能处理数字，Tokenizer 是文本和数字之间的桥梁。我们会从零实现 BPE 算法，还会了解 Tokenizer 导致的经典「坑」，比如为什么 GPT 数不清 "strawberry" 里有几个 r。

👀 输出要点
- Cell 20：核心概念图谱（ASCII 架构图）、关键公式速查表（5 行）、三道面试题与参考答案、本章要点回顾（三大板块：位置编码、GPT 架构、采样策略）
- Cell 21：三道 Extra 思考题（权重共享、Temperature 数学、Sinusoidal vs RoPE）
- Cell 22：下一章链接和延伸阅读（GPT-1 论文、RoFormer 论文）

❓ 预判问题

Q: GPT 怎么训练？loss 函数是什么？
A: 就是标准的 cross_entropy——给定前 t 个 token，预测第 t+1 个 token。Cell 14 的 forward 方法里已经写了：`F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))`。训练数据就是大量文本，targets 是 input 向右移一位。

Q: 这个迷你 GPT 能生成有意义的文本吗？
A: 要训练之后才行。vocab_size=1000 太小（GPT-2 是 50,257），4 层也太浅。但如果在一个小数据集（如莎士比亚全集）上训练，是可以生成模仿风格的文本的。Karpathy 的 nanoGPT 就是类似的实验。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，回顾前置知识，运行环境 | 0-5 |
| 8 | 正弦位置编码理论 + 可视化 | 6-7 |
| 25 | RoPE 理论 + 代码 + 可视化 + 对比 | 8-11 |
| 45 | **休息** | -- |
| 50 | GPT 完整模型组装 | 12-14 |
| 65 | 文本生成采样策略 | 15-19 |
| 82 | **休息** | -- |
| 87 | 练习：Temperature + Top-k 采样 | 23 |
| 95 | 总结 + 下一章预告 | 20-22 |
| 100 | 结束 | -- |

---

## 附录 B：关键数据快速参考

### 核心公式

| 公式 | 表达式 | 说明 |
|:---|:---|:---|
| 正弦位置编码 | PE(pos,2i) = sin(pos/10000^{2i/d}), PE(pos,2i+1) = cos(pos/10000^{2i/d}) | 固定绝对位置编码 |
| RoPE 旋转 | q' = R(m*theta) * q，点积 = q^T R(n-m) k | 只依赖相对位置 |
| RoPE 频率 | theta_i = 1/10000^{2i/d} | 低维快，高维慢 |
| Temperature | p_i = exp(z_i/T) / sum(exp(z_j/T)) | T->0 贪心，T->inf 均匀 |
| Top-p | 保留最小集合 S 使 sum(p_i) >= p | 动态候选集 |

### 张量维度速查

| 变量 | 形状 | 来源 |
|:---|:---|:---|
| 位置编码矩阵 | [1, max_len, d_model] | Cell 7: [1, 5000, 64] |
| RoPE 频率 | [max_seq_len, dim/2] | Cell 9: [100, 32] |
| GPT 输入 idx | [B, T] | Cell 14: token IDs |
| Token Embedding | [vocab_size, n_embd] | Cell 14: [1000, 128] |
| Position Embedding | [block_size, n_embd] | Cell 14: [128, 128] |
| Transformer 输出 | [B, T, n_embd] | Cell 14: [B, T, 128] |
| Logits | [B, T, vocab_size] | Cell 14: [B, T, 1000] |
| Temperature 可视化 logits | [7] | Cell 17: [2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0] |

### 关键数值

| 数值 | 来源 | 含义 |
|:---|:---|:---|
| GPT 参数量 937,728 | Cell 14 | 4层4头128维的迷你GPT |
| 初始序列 [607, 484, 901, 954, 177] | Cell 19 | 随机生成的5个初始token |
| Greedy 输出含 678, 200, 588... | Cell 19 | temp=0.1 的确定性输出 |
| Top-k=3 输出含重复 701, 496 | Cell 19 | 候选集太小导致重复 |
| 练习 temp=0.3 几乎全选 "the" | Cell 23 | 低温确定性验证 |
| 练习 top_k=3 只选 the/cat/sat | Cell 23 | Top-k 过滤验证 |

---

## 附录 C：应急预案

### 场景 1：环境问题

**症状：** Cell 4 报错 `ModuleNotFoundError: No module named 'torch'`

**应对：**
1. 在终端运行 `pip install torch torchvision`
2. 如果是 conda 环境：`conda install pytorch -c pytorch`
3. 备选：切换到 Google Colab，上传 notebook

### 场景 2：中文字体不显示

**症状：** matplotlib 图表中文显示为方框

**应对：**
1. Cell 4 已配置 `plt.rcParams["font.sans-serif"]` 备选字体列表
2. Cell 5 额外加载了 `../assets/fonts/NotoSansCJKsc-Regular.otf`
3. 如果仍有问题：`plt.rcParams['font.sans-serif'] = ['DejaVu Sans']`（牺牲中文，保证图能看）
4. 口头补充中文标签含义

### 场景 3：Cell 5 架构全景图渲染缓慢或报错

**应对：**
1. 这张图代码较长，如果报错检查字体路径 `../assets/fonts/NotoSansCJKsc-Regular.otf` 是否存在
2. 如果字体文件不存在，注释掉 `fm.fontManager.addfont(font_path)` 那行
3. 实在不行就跳过这个 Cell，在白板上画简化版架构图

### 场景 4：Cell 7 或 Cell 9 的 TODO 学生卡住

**应对：**
1. Cell 7 的 TODO 极简——cos 版本和 sin 版本输入完全相同，只换函数名。给 2 分钟后直接展示
2. Cell 9 有两个 TODO，xk_ 和 xk_out 与 xq_/xq_out 完全对称。提示「看上面 xq_ 的写法，把 xq 换成 xk」
3. 源 notebook 中 TODO 后面已有参考答案

### 场景 5：时间不够

**可跳过的内容（按优先级）：**
1. Cell 11 Sin/Cos vs RoPE 数学对比（改为口头一句话总结）- 省 5 分钟
2. Cell 10 RoPE 可视化（口头描述「低维快高维慢」即可）- 省 3 分钟
3. Cell 18 Top-k/Top-p 可视化（Cell 17 的 Temperature 可视化已足够说明采样原理）- 省 3 分钟

**不可跳过的核心：**
- Cell 7：正弦位置编码 + 可视化（位置编码是本章第一个核心概念）
- Cell 14：GPT 类定义（本章的标题就是「GPT 组装」）
- Cell 17 + Cell 19：Temperature 可视化 + 生成演示（采样策略是本章第二个核心概念）
- Cell 23：练习（动手环节不可省）

### 场景 6：学生提出超纲问题（如 KV Cache、FlashAttention、RLHF）

**应对：**
1. KV Cache：简要说明「生成时每轮只需要算新 token 的 Q，历史 token 的 K/V 缓存起来复用，避免重复计算」
2. FlashAttention：「分块计算 Attention，避免存储完整的 n*n 矩阵，减少内存访问」
3. RLHF：「GPT 预训练后还需要用人类反馈来对齐，这是后续课程的内容」
4. 记录在白板上，不展开以免偏离主线